# 9 — E5: Composition (Optional)

**Hypothesis:** Circuits for structurally related concepts compose linearly
or produce emergent interaction neurons.

Compares the circuit for a composition pair (A+B in same prompt) against the
union of individual circuits to find compositional residuals.

**Prerequisites:** `1D_composition_prompts.ipynb` must be run first.

In [ ]:
# Cell 1 – Configuration
import subprocess, sys, os
for pkg in ["h5py", "numpy", "pandas", "matplotlib", "seaborn"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)

import numpy as np
import pandas as pd
import h5py
import matplotlib.pyplot as plt
import seaborn as sns

MODEL_CONFIGS = {
    "QW": {"id": "Qwen/Qwen2.5-Coder-7B",                "n_layers": 28, "mlp_dim": 3584},
    "DS": {"id": "deepseek-ai/deepseek-coder-6.7b-base",  "n_layers": 32, "mlp_dim": 4096},
}
COMBOS = [
    {"lang": "P", "model": "QW", "label": "Python/Qwen"},
    {"lang": "R", "model": "QW", "label": "Rust/Qwen"},
]

EPSILON = 0.001
CONSISTENCY = 0.8
N_LAYERS = max(c["n_layers"] for c in MODEL_CONFIGS.values())

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import shutil
    mp = "/content/drive"
    subprocess.run(["fusermount", "-uz", mp], capture_output=True)
    if os.path.isdir(mp):
        shutil.rmtree(mp, ignore_errors=True)
    drive.mount(mp)
    DATA_DIR = "/content/drive/MyDrive/DATA/New-Atlas"
else:
    DATA_DIR = "/Users/piotrwilam/Data/New-Atlas"



print(f"Settings: eps={EPSILON}, cons={CONSISTENCY}")

In [ ]:
# Cell 2 – Composition pairs

PYTHON_PAIRS = [
    ("ast__For", "ast__Break"),
    ("ast__For", "ast__Continue"),
    ("ast__Try", "ast__Raise"),
    ("ast__FunctionDef", "ast__Return"),
    ("ast__FunctionDef", "ast__Yield"),
    ("ast__If", "ast__Assign"),
    ("ast__While", "ast__Break"),
    ("ast__ClassDef", "ast__FunctionDef"),
]

RUST_PAIRS = [
    ("rust__Fn", "rust__Return"),
    ("rust__Match", "rust__If"),
    ("rust__Impl", "rust__Trait"),
    ("rust__For", "rust__Break"),
    ("rust__Loop", "rust__Break"),
    ("rust__Struct", "rust__Impl"),
    ("rust__Enum", "rust__Match"),
    ("rust__Async", "rust__Await"),
]

LANG_PAIRS = {"P": PYTHON_PAIRS, "R": RUST_PAIRS}

print(f"Python pairs: {len(PYTHON_PAIRS)}")
print(f"Rust pairs: {len(RUST_PAIRS)}")

In [ ]:
# Cell 3 – Load individual and composition masks

individual_masks = {}   # (lang, model) -> {concept: {layer: mask}}
composition_masks = {}  # (lang, model) -> {concept: {layer: mask}}

for combo in COMBOS:
    prefix = f"{combo['lang']}_{combo['model']}_"
    key = (combo["lang"], combo["model"])

    # Individual universal masks
    ind_path = os.path.join(DATA_DIR, f"{prefix}3_object_masks_eps{EPSILON}_cons{CONSISTENCY}.h5")
    if os.path.exists(ind_path):
        ind = {}
        with h5py.File(ind_path, "r") as f:
            if "universal_masks" in f:
                um = f["universal_masks"]
                for lid in range(N_LAYERS):
                    lk = f"layer_{lid}"
                    if lk not in um:
                        continue
                    for ds_name in um[lk]:
                        if ds_name not in ind:
                            ind[ds_name] = {}
                        ind[ds_name][lid] = np.array(um[lk][ds_name], dtype=bool)
        individual_masks[key] = ind
        print(f"  {combo['label']} individual: {len(ind)} concepts")
    else:
        print(f"  WARNING: missing individual masks for {combo['label']}")

    # Composition masks
    comp_path = os.path.join(DATA_DIR, f"{prefix}3_composition_masks_eps{EPSILON}_cons{CONSISTENCY}.h5")
    if os.path.exists(comp_path):
        comp = {}
        with h5py.File(comp_path, "r") as f:
            if "universal_masks" in f:
                um = f["universal_masks"]
                for lid in range(N_LAYERS):
                    lk = f"layer_{lid}"
                    if lk not in um:
                        continue
                    for ds_name in um[lk]:
                        if ds_name not in comp:
                            comp[ds_name] = {}
                        comp[ds_name][lid] = np.array(um[lk][ds_name], dtype=bool)
        composition_masks[key] = comp
        print(f"  {combo['label']} composition: {len(comp)} concepts")
    else:
        print(f"  WARNING: missing composition masks for {combo['label']} — run 1D first")

In [ ]:
# Cell 4 – Compute compositional residuals

results = []

for key in individual_masks:
    if key not in composition_masks:
        continue
    lang, model = key
    ind = individual_masks[key]
    comp = composition_masks[key]
    pairs = LANG_PAIRS.get(lang, [])

    for concept_a, concept_b in pairs:
        # The composition key in HDF5 — depends on how 1D names the pair
        comp_key = f"{concept_a}+{concept_b}"
        if comp_key not in comp:
            comp_key = f"{concept_b}+{concept_a}"  # try reversed
        if comp_key not in comp:
            continue

        for lid in range(N_LAYERS):
            UA = ind.get(concept_a, {}).get(lid)
            UB = ind.get(concept_b, {}).get(lid)
            C = comp[comp_key].get(lid)

            if UA is None or UB is None or C is None:
                continue

            union = UA | UB
            residual = C & ~union         # neurons only in composition
            missing = union & ~C           # individual neurons absent from composition

            results.append({
                "lang": lang, "model": model,
                "concept_a": concept_a, "concept_b": concept_b,
                "layer": lid,
                "n_composition": int(C.sum()),
                "n_union": int(union.sum()),
                "n_residual": int(residual.sum()),
                "n_missing": int(missing.sum()),
                "residual_fraction": int(residual.sum()) / int(C.sum()) if C.sum() > 0 else 0.0,
                "missing_fraction": int(missing.sum()) / int(union.sum()) if union.sum() > 0 else 0.0,
            })

df_results = pd.DataFrame(results)
print(f"Results: {len(df_results)} rows")

if len(df_results) > 0:
    print(f"Mean residual fraction: {df_results['residual_fraction'].mean():.3f}")
    print(f"Mean missing fraction: {df_results['missing_fraction'].mean():.3f}")
else:
    print("No results — composition masks not yet available. Run 1D first.")

In [ ]:
# Cell 5 – Residual heatmap

if len(df_results) > 0:
    for (lang, model), sub in df_results.groupby(["lang", "model"]):
        pair_labels = sub.apply(lambda r: f"{r['concept_a'].split('__')[1]}+{r['concept_b'].split('__')[1]}", axis=1)
        unique_pairs = sorted(pair_labels.unique())
        pivot = sub.copy()
        pivot["pair"] = pair_labels
        hm = pivot.pivot_table(index="pair", columns="layer", values="residual_fraction", aggfunc="mean")
        hm = hm.reindex(unique_pairs)

        fig, ax = plt.subplots(figsize=(14, max(4, len(unique_pairs)*0.5)))
        sns.heatmap(hm, cmap="YlOrRd", ax=ax, vmin=0)
        ax.set_title(f"Compositional residual fraction — {lang}_{model}")
        ax.set_xlabel("Layer")
        plt.tight_layout()
        plt.savefig(os.path.join(DATA_DIR, f"8_E5_residual_heatmap_{lang}_{model}.png"), dpi=150)
        plt.show()
else:
    print("No data to plot.")

In [ ]:
# Cell 6 – Save

if len(df_results) > 0:
    out_path = os.path.join(DATA_DIR, "8_E5_composition_results.csv")
    df_results.to_csv(out_path, index=False)
    print(f"Saved: {out_path}")

print("\n9 complete.")